# 코드 설명

- **주제** : 패킹(압축·난독화) 여부와 패커 종류를 바이트 N-그램 특성 + 랜덤 포레스트로 분류하기
- **사용 데이터** : 정상 PE 실행파일 3종 모음 (아래 표). **용량이 커서 저장소에는 포함하지 않으며, 준비 셀로 내려받습니다.**

    | 폴더 | 레이블 | 설명 |
    |---|---|---|
    | `Benign PE Samples` | 0 | 패킹하지 않은 원본 |
    | `Benign PE Samples UPX` | 1 | UPX로 패킹 |
    | `Benign PE Samples Amber` | 2 | Amber로 패킹 |

- **사용 라이브러리**
    - `scikit-learn` — 학습/테스트 분할, 랜덤 포레스트, 혼동 행렬
    - 표준 라이브러리 `collections` — N-그램 빈도 집계
- **실행 순서** : 파일 목록·레이블 수집 → 학습/테스트 분할 → 최빈 N-그램 선정 → 특성 벡터화 → 랜덤 포레스트 학습 → 혼동 행렬 평가

> **최신화 메모**
>
> 원본은 `from nltk import ngrams` 로 N-그램을 만들었습니다. 이 한 줄 때문에 무거운 `nltk` 패키지를 설치해야 했는데, N-그램 생성은 파이썬 **표준 라이브러리만으로** 똑같이 할 수 있습니다. 그래서 `nltk` 의존성을 제거하고 `zip` 기반 함수로 대체했습니다.
>
> ```python
> # 원본
> from nltk import ngrams
> list(ngrams(byte_sequence, N))
> # 현재 (동일 결과, 의존성 없음)
> list(zip(*[byte_sequence[i:] for i in range(N)]))
> ```
>
> scikit-learn API(`train_test_split`, `RandomForestClassifier`, `confusion_matrix`)는 1.5+ 에서 그대로 동작하므로 코드 변경이 없습니다.

In [1]:
# ── Google Colab 준비 셀 ─────────────────────────────────────────
# 로컬(VS Code, JupyterLab)에서 실행하면 데이터가 이미 있는지 확인만 합니다.
# Colab에서 실행하면 저장소를 내려받고, 아래 데이터_URL 에서 PE 샘플을 받습니다.
import os
import subprocess
import sys

# ★ PE 샘플(약 440MB)을 올려둔 곳의 "직접 다운로드" 주소(zip)를 넣으세요.
#   예: GitHub Release 자산, 구글 드라이브 공유 링크(gdown) 등.
#   비워 두면 다운로드를 건너뛰고, 데이터가 없을 때 안내만 출력합니다.
데이터_URL = ""

데이터폴더 = ["Benign PE Samples", "Benign PE Samples UPX", "Benign PE Samples Amber"]

if "google.colab" in sys.modules:
    저장소 = "https://github.com/isl-hedgehog/python-cyber-seminar.git"
    이름 = 저장소.rsplit("/", 1)[-1].removesuffix(".git")
    실습폴더 = "02-adversarial-malware"
    if not os.path.isdir(f"/content/{이름}"):
        subprocess.run(["git", "clone", "-q", 저장소], cwd="/content", check=True)
    os.chdir(f"/content/{이름}/{실습폴더}")
    print("Colab 준비 완료 :", os.getcwd())

데이터_있음 = all(os.path.isdir(d) for d in 데이터폴더)
if 데이터_있음:
    print("PE 샘플 데이터 확인됨 — 다운로드를 건너뜁니다.")
elif 데이터_URL:
    print("PE 샘플 데이터를 내려받습니다 ... (약 440MB, 시간이 걸립니다)")
    subprocess.run(["wget", "-q", "-O", "pe_samples.zip", 데이터_URL], check=True)
    subprocess.run(["unzip", "-q", "-o", "pe_samples.zip"], check=True)
    os.remove("pe_samples.zip")
    print("다운로드·압축 해제 완료")
else:
    print("⚠ PE 샘플 데이터가 없습니다.")
    print("  위 셀의 데이터_URL 에 zip 다운로드 주소를 넣고 다시 실행하거나,")
    print("  로컬에서는 아래 3개 폴더를 이 노트북과 같은 위치에 두세요:")
    for d in 데이터폴더:
        print("   -", d)

PE 샘플 데이터 확인됨 — 다운로드를 건너뜁니다.


### 1. 파일 목록과 레이블 수집

세 폴더를 돌며 각 파일의 경로를 모으고, 폴더에 해당하는 레이블(0/1/2)을 함께 기록합니다. 분류기는 "이 파일이 어느 폴더(패킹 방식)에서 왔는가"를 맞히도록 학습됩니다.

In [2]:
import os
from os import listdir

directories_with_labels = [
    ("Benign PE Samples", 0),
    ("Benign PE Samples UPX", 1),
    ("Benign PE Samples Amber", 2),
]
list_of_samples = []
labels = []
for dataset_path, label in directories_with_labels:
    samples = [f for f in listdir(dataset_path)]
    for file in samples:
        file_path = os.path.join(dataset_path, file)
        list_of_samples.append(file_path)
        labels.append(label)

print("수집한 파일 수:", len(list_of_samples))

수집한 파일 수: 651


### 2. 학습/테스트 분할

`stratify=labels` 로 세 레이블의 비율을 학습·테스트 양쪽에서 동일하게 유지합니다. `random_state=11` 로 분할을 고정해 실행마다 같은 결과가 나오게 합니다.

In [3]:
from sklearn.model_selection import train_test_split

samples_train, samples_test, labels_train, labels_test = train_test_split(
    list_of_samples, labels, test_size=0.3, stratify=labels, random_state=11
)

### 3. N-그램 특성 함수 정의

바이트 N-그램은 "연속한 N개 바이트의 조합"입니다. 패커마다 코드·데이터를 배치·압축하는 방식이 달라, 자주 등장하는 바이트 조합의 분포가 패커를 구분하는 단서가 됩니다.

- `read_file` — 파일을 바이트로 읽음
- `byte_sequence_to_Ngrams` — **nltk 없이** `zip` 으로 N-그램 리스트 생성
- `extract_Ngram_counts` — 한 파일의 N-그램 개수를 `Counter` 로 집계
- `featurize_sample` — 미리 고른 N-그램 K1개의 등장 횟수를 특성 벡터로 반환

In [4]:
import collections
import numpy as np


def read_file(file_path):
    """ 이진 파일의 바이트열을 읽는다. """
    with open(file_path, "rb") as binary_file:
        data = binary_file.read()
    return data


def byte_sequence_to_Ngrams(byte_sequence, N):
    """ 바이트열에서 N-그램 리스트를 만든다 (표준 라이브러리만 사용). """
    return list(zip(*[byte_sequence[i:] for i in range(N)]))


def extract_Ngram_counts(file, N):
    """ 이진 파일을 읽어 N-그램별 개수를 센다. """
    filebyte_sequence = read_file(file)
    file_Ngrams = byte_sequence_to_Ngrams(filebyte_sequence, N)
    return collections.Counter(file_Ngrams)


def featurize_sample(sample, K1_most_frequent_Ngrams_list):
    """ 샘플의 특성 벡터를 만든다. 특성은 우리가 선택한 N-그램 K1개의 개수이다. """
    K1 = len(K1_most_frequent_Ngrams_list)
    feature_vector = K1 * [0]
    file_Ngrams = extract_Ngram_counts(sample, N)
    for i in range(K1):
        feature_vector[i] = file_Ngrams[K1_most_frequent_Ngrams_list[i]]
    return feature_vector

### 4. 최빈 N-그램 K1개 선정

학습 파일 전체에서 N=2(바이트 2개 조합)의 등장 횟수를 모두 합산한 뒤, 가장 자주 나온 상위 100개(K1)를 특성으로 선택합니다. 이 100개가 특성 벡터의 각 칸(차원)이 됩니다.

> 학습 파일을 한 번 전부 읽으므로 수십 초가 걸릴 수 있습니다.

In [5]:
N = 2
total_Ngram_count = collections.Counter([])
for file in samples_train:
    total_Ngram_count += extract_Ngram_counts(file, N)
K1 = 100
K1_most_common_Ngrams = total_Ngram_count.most_common(K1)
K1_most_common_Ngrams_list = [x[0] for x in K1_most_common_Ngrams]
print("선택한 특성(N-그램) 개수:", len(K1_most_common_Ngrams_list))

선택한 특성(N-그램) 개수: 100


### 5. 학습 데이터 특성 벡터화

각 학습 파일을 길이 100짜리 특성 벡터로 변환합니다. `X_train` 은 (파일 수 × 100) 형태가 됩니다.

In [6]:
Ngram_features_list_train = []
y_train = []
for i in range(len(samples_train)):
    file = samples_train[i]
    NGram_features = featurize_sample(file, K1_most_common_Ngrams_list)
    Ngram_features_list_train.append(NGram_features)
    y_train.append(labels_train[i])
X_train = Ngram_features_list_train

### 6. 랜덤 포레스트 학습

100그루의 결정 트리를 학습시킵니다. 각 트리는 특성(N-그램 빈도) 일부를 보고 레이블을 투표하며, 다수결로 최종 예측을 냅니다.

In [7]:
from sklearn.ensemble import RandomForestClassifier

clf = RandomForestClassifier(n_estimators=100)
clf = clf.fit(X_train, y_train)

### 7. 테스트 데이터 특성 벡터화

학습 때 고른 **같은** N-그램 100개 기준으로 테스트 파일도 벡터화합니다. 특성 기준이 학습과 동일해야 비교가 성립합니다.

In [8]:
Ngram_features_list_test = []
y_test = []
for i in range(len(samples_test)):
    file = samples_test[i]
    NGram_features = featurize_sample(file, K1_most_common_Ngrams_list)
    Ngram_features_list_test.append(NGram_features)
    y_test.append(labels_test[i])
X_test = Ngram_features_list_test

### 8. 예측과 혼동 행렬 평가

테스트 파일을 예측하고 혼동 행렬로 결과를 봅니다. 행은 실제 레이블, 열은 예측 레이블이며, 대각선 값이 클수록 정확히 맞힌 것입니다(순서: 0 원본, 1 UPX, 2 Amber).

In [9]:
y_pred = clf.predict(X_test)

In [10]:
from sklearn.metrics import confusion_matrix

confusion_matrix(y_test, y_pred)

array([[113,   1,   0],
       [  0,  60,   0],
       [  0,   0,  22]])